# Show, Attend and Tell — Image Captioning

This notebook trains an image captioning model on the Flickr8k dataset.

**Runtime:** Go to `Runtime > Change runtime type` and select **GPU** (T4 is fine).

## 1. Clone the repo and install dependencies

In [ ]:
# !git clone https://github.com/HunterBaker01/S_A_T.git
# %cd S_A_T

!pip install -q h5py tqdm pillow

## 2. Mount Google Drive and link dataset

Update the `DRIVE_DATA_PATH` below to point to wherever your Flickr8k dataset lives on your Drive.

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")

# ── Change this to your dataset's folder on Google Drive ──
DRIVE_DATA_PATH = "/content/drive/MyDrive/flickr8k"

# Symlink it to the local "data" for convenience
os.symlink(DRIVE_DATA_PATH, "data")

print("Contents of data/:", os.listdir("data"))
print(f"Number of images: {len([f for f in os.listdir('data/Images') if f.endswith('.jpg')])}")

## 3. Build vocabulary and extract features

In [ ]:
import torch
from get_data import build_vocab, get_loaders, Flickr8kDataset
from models import MyVGG16, AttentionDecoder
from create_h5 import save_to_h5

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# Paths
IMAGES_DIR = "data/Images"
TOKENS_FILE = "data/captions.txt"
HDF5_FILE = "data/features.h5"
VOCAB_PATH = "vocab.pkl"

# Hyperparameters
EMBED_DIM = 256
HIDDEN_DIM = 512
ATTENTION_DIM = 256
ENCODER_DIM = 512
LEARNING_RATE = 1e-3
BATCH_SIZE = 64
EPOCHS = 10
MIN_WORD_FREQ = 2

In [ ]:
# Build vocabulary and split train/test
train_dict, test_dict, vocab = build_vocab(TOKENS_FILE, MIN_WORD_FREQ, VOCAB_PATH)
vocab_size = len(vocab)
print(f"Train images: {len(train_dict)}, Test images: {len(test_dict)}")

In [ ]:
# Extract VGG16 features (takes ~5 min on T4 GPU, skipped if features.h5 exists)
if not os.path.isfile(HDF5_FILE):
    encoder = MyVGG16().to(DEVICE)
    save_to_h5(encoder, IMAGES_DIR, DEVICE, HDF5_FILE)
    del encoder
    torch.cuda.empty_cache()
else:
    print(f"Features already cached at {HDF5_FILE}, skipping extraction.")

## 4. Train the model

In [ ]:
from torch import nn, optim
from eval import train_epoch, validate

# Data loaders
train_loader, test_loader = get_loaders(
    train_dict, test_dict, vocab,
    h5_path=HDF5_FILE,
    batch_size=BATCH_SIZE,
    num_workers=2,
    collate_fn=Flickr8kDataset.collate_fn,
)

# Attention decoder
decoder = AttentionDecoder(
    embedding_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    vocab_size=vocab_size,
    encoder_dim=ENCODER_DIM,
    attention_dim=ATTENTION_DIM,
    dropout=0.5,
).to(DEVICE)

optimizer = optim.Adam(decoder.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss(ignore_index=0)

print(f"Decoder parameters: {sum(p.numel() for p in decoder.parameters()):,}")

In [ ]:
# Training loop
best_val_loss = float("inf")
train_losses, val_losses = [], []

for epoch in range(EPOCHS):
    train_loss = train_epoch(decoder, train_loader, criterion, optimizer, vocab_size, epoch, DEVICE)
    val_loss = validate(decoder, test_loader, criterion, vocab_size, DEVICE)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{EPOCHS} — Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(decoder.state_dict(), "best_checkpoint.pth")
        print(f"  -> Best model saved (val_loss={val_loss:.4f})")

In [ ]:
# Plot training curves
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(range(1, EPOCHS + 1), train_losses, label="Train Loss")
plt.plot(range(1, EPOCHS + 1), val_losses, label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Progress")
plt.legend()
plt.grid(True)
plt.show()

## 5. Generate captions and visualize attention

In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from matplotlib import cm


def caption_image(image_id, decoder, vocab, h5_path, images_dir, show_attention=True):
    """
    Generate a caption for a single image with optional attention visualization.

    Loads the precomputed spatial features (49, 512), runs the decoder in
    autoregressive mode, and optionally overlays attention heat-maps on the
    original image for each generated word.
    """
    decoder.eval()

    # Load spatial features (num_pixels, encoder_dim)
    with h5py.File(h5_path, "r") as h5f:
        features = torch.tensor(h5f[image_id][:], dtype=torch.float32)
    features = features.unsqueeze(0).to(DEVICE)  # (1, 49, 512)

    with torch.no_grad():
        token_ids, alphas = decoder.generate(features, max_len=20)

    # Decode token IDs to words
    words = []
    word_alphas = []
    for i, idx in enumerate(token_ids[0].cpu().tolist()):
        word = vocab.toks.get(idx, "<unk>")
        if word == "eos":
            break
        if word not in ("pad", "sos"):
            words.append(word)
            word_alphas.append(alphas[0, i].cpu().numpy())
    caption = " ".join(words)

    # Load original image
    img_path = os.path.join(images_dir, image_id + ".jpg")
    img = Image.open(img_path).convert("RGB")

    if show_attention and word_alphas:
        # Show the image + one attention map per generated word
        n_words = len(words)
        cols = min(5, n_words + 1)
        rows = (n_words + 1 + cols - 1) // cols  # ceil division
        fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
        if rows == 1 and cols == 1:
            axes = np.array([axes])
        axes = axes.flatten()

        # First panel: original image with caption
        axes[0].imshow(img)
        axes[0].set_title("original", fontsize=10)
        axes[0].axis("off")

        # Subsequent panels: attention overlay per word
        for i, (word, alpha) in enumerate(zip(words, word_alphas)):
            ax = axes[i + 1]
            ax.imshow(img)
            # Reshape attention (49,) -> (7,7) and upscale to image size
            alpha_map = alpha.reshape(7, 7)
            ax.imshow(
                alpha_map, alpha=0.6, extent=(0, img.size[0], img.size[1], 0),
                interpolation="bilinear", cmap="jet",
            )
            ax.set_title(word, fontsize=10)
            ax.axis("off")

        # Hide unused axes
        for j in range(n_words + 1, len(axes)):
            axes[j].axis("off")

        fig.suptitle(caption, fontsize=13, y=1.02)
        plt.tight_layout()
        plt.show()
    else:
        plt.figure(figsize=(6, 6))
        plt.imshow(img)
        plt.title(caption, fontsize=12, wrap=True)
        plt.axis("off")
        plt.show()

    return caption

In [ ]:
# Load best model and caption a few test images
decoder.load_state_dict(torch.load("best_checkpoint.pth", map_location=DEVICE))

test_image_ids = list(test_dict.keys())[:5]
for img_id in test_image_ids:
    generated = caption_image(img_id, decoder, vocab, HDF5_FILE, IMAGES_DIR)
    print(f"Generated: {generated}")
    print(f"Reference: {test_dict[img_id][0]}")
    print("-" * 60)